
### Step 1: Inspect the Dataset

The purpose of this step is to inspect the first 100 rows of the NYC Yellow Taxi dataset.

Each row represents a taxi trip. The dataset contains information about the trip, including pickup and dropoff timestamps, trip distance, fare amount, passenger count, and payment details.

The `head()` function displays the first five rows, `dtypes` shows the data type of each column, and `shape` reports the number of rows and columns in the inspected sample.

This initial inspection helps us understand the dataset structure before performing the full data profiling and cleaning steps.



In [1]:

import pandas as pd

url = (
    "https://github.com/DataTalksClub/nyc-tlc-data/"
    "releases/download/yellow/yellow_tripdata_2021-01.csv.gz"
)

data = pd.read_csv(url, nrows=100)

print("First 5 rows:")
print(data.head())

print("\nColumn data types:")
print(data.dtypes)

print("\nDataset shape:")
print(data.shape)


First 5 rows:
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3          10.60           1                  N           138           132   
4           4.94           1                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amount  to

### Step 2: Profile the Entire Dataset in Chunks

The purpose of this step is to perform an initial profile of the entire NYC Yellow Taxi dataset.

Since the dataset contains a large number of rows, it is read in chunks of 100,000 rows using pandas. This reduces memory usage compared with loading the entire dataset at once.

For each chunk, the code calculates the total number of rows, counts missing values in each column, and tracks the minimum and maximum values. The data types are also recorded to understand the structure of the dataset.

The results are combined into a profiling table that summarizes the dataset's structure and value ranges. This information helps identify potential data quality issues before performing the detailed analysis and cleaning steps.

In [2]:

# Step 2: Profile the entire dataset in chunks

import pandas as pd

url = (
    "https://github.com/DataTalksClub/nyc-tlc-data/"
    "releases/download/yellow/yellow_tripdata_2021-01.csv.gz"
)

chunk_size = 100_000

dtype = {
    "VendorID": "Int64",
    "passenger_count": "Int64",
    "trip_distance": "float64",
    "RatecodeID": "Int64",
    "store_and_fwd_flag": "string",
    "PULocationID": "Int64",
    "DOLocationID": "Int64",
    "payment_type": "Int64",
    "fare_amount": "float64",
    "extra": "float64",
    "mta_tax": "float64",
    "tip_amount": "float64",
    "tolls_amount": "float64",
    "improvement_surcharge": "float64",
    "total_amount": "float64",
    "congestion_surcharge": "float64",
}

parse_dates = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
]

total_rows = 0
null_counts = None
column_types = None
column_min = {}
column_max = {}

for chunk in pd.read_csv(
    url,
    chunksize=chunk_size,
    dtype=dtype,
    parse_dates=parse_dates,
):

    total_rows += len(chunk)

    if column_types is None:
        column_types = chunk.dtypes.astype(str)

        null_counts = pd.Series(
            0,
            index=chunk.columns,
            dtype="int64",
        )

    null_counts = null_counts.add(
        chunk.isna().sum(),
        fill_value=0,
    ).astype("int64")

    for column in chunk.columns:

        values = chunk[column].dropna()

        if values.empty:
            continue

        current_min = values.min()
        current_max = values.max()

        if (
            column not in column_min
            or current_min < column_min[column]
        ):
            column_min[column] = current_min

        if (
            column not in column_max
            or current_max > column_max[column]
        ):
            column_max[column] = current_max

profile = pd.DataFrame({
    "dtype": column_types,
    "null_count": null_counts,
    "null_percentage": (
        null_counts / total_rows * 100
    ).round(2),
    "min": pd.Series(column_min),
    "max": pd.Series(column_max),
})

print("Total rows:", total_rows)
print("Total columns:", len(column_types))
print(profile.to_string())

Total rows: 1369765
Total columns: 18
                                dtype  null_count  null_percentage                  min                  max
VendorID                        Int64       98352             7.18                    1                    2
tpep_pickup_datetime   datetime64[us]           0             0.00  2008-12-31 23:05:14  2021-02-22 16:52:16
tpep_dropoff_datetime  datetime64[us]           0             0.00  2008-12-31 23:07:22  2021-02-22 16:56:15
passenger_count                 Int64       98352             7.18                    0                    8
trip_distance                 float64           0             0.00                  0.0            263163.28
RatecodeID                      Int64       98352             7.18                    1                   99
store_and_fwd_flag             string       98352             7.18                    N                    Y
PULocationID                    Int64           0             0.00                    1   

### Step 3: Find Data Quality Problems

The purpose of this step is to identify potential data quality problems in the NYC Yellow Taxi dataset before cleaning it.

The dataset is processed in chunks of 100,000 rows. For each chunk, the code checks for missing values, invalid or out-of-range pickup and dropoff dates, negative or zero fares, negative or zero trip distances, and zero passenger counts.

The code also uses row hashes to identify potential duplicate rows across chunks. The results are accumulated to provide a summary of the data quality issues found in the entire dataset.

These findings help determine which records should be removed, corrected, or flagged for further investigation during the cleaning step.

In [3]:

# Step 3: Find data quality problems

import pandas as pd

url = (
    "https://github.com/DataTalksClub/nyc-tlc-data/"
    "releases/download/yellow/yellow_tripdata_2021-01.csv.gz"
)

chunk_size = 100_000

total_rows = 0
null_counts = None

pickup_outside_january = 0
dropoff_outside_january = 0
invalid_pickup_dates = 0
invalid_dropoff_dates = 0

negative_fares = 0
zero_fares = 0
negative_distances = 0
zero_distances = 0
zero_passengers = 0

# Track row hashes across chunks to detect duplicates
seen_hashes = set()
duplicate_rows = 0

for chunk in pd.read_csv(
    url,
    chunksize=chunk_size,
    dtype={
        "VendorID": "Int64",
        "passenger_count": "Int64",
        "RatecodeID": "Int64",
        "store_and_fwd_flag": "string",
        "PULocationID": "Int64",
        "DOLocationID": "Int64",
        "payment_type": "Int64",
    }
):
    total_rows += len(chunk)

    # Count missing values across all chunks
    if null_counts is None:
        null_counts = pd.Series(
            0, index=chunk.columns, dtype="int64"
        )

    null_counts = null_counts.add(
        chunk.isna().sum(),
        fill_value=0
    ).astype("int64")

    # Parse dates for reliable date comparisons
    pickup = pd.to_datetime(
        chunk["tpep_pickup_datetime"], errors="coerce"
    )
    dropoff = pd.to_datetime(
        chunk["tpep_dropoff_datetime"], errors="coerce"
    )

    invalid_pickup_dates += int(pickup.isna().sum())
    invalid_dropoff_dates += int(dropoff.isna().sum())

    pickup_outside_january += int(
        (pickup.notna() &
         ((pickup < "2021-01-01") | (pickup >= "2021-02-01"))).sum()
    )

    dropoff_outside_january += int(
        (dropoff.notna() &
         ((dropoff < "2021-01-01") | (dropoff >= "2021-02-01"))).sum()
    )

    # Fare and distance checks
    negative_fares += int((chunk["fare_amount"] < 0).sum())
    zero_fares += int((chunk["fare_amount"] == 0).sum())

    negative_distances += int((chunk["trip_distance"] < 0).sum())
    zero_distances += int((chunk["trip_distance"] == 0).sum())

    # Passenger count check
    zero_passengers += int(
        (chunk["passenger_count"] == 0).sum()
    )

    # Detect exact-row duplicate candidates across chunks
    row_hashes = pd.util.hash_pandas_object(
        chunk, index=False
    )

    for row_hash in row_hashes:
        hash_value = int(row_hash)

        if hash_value in seen_hashes:
            duplicate_rows += 1
        else:
            seen_hashes.add(hash_value)

print("Total rows checked:", total_rows)

print("\nDate checks")
print("Pickup outside January:", pickup_outside_january)
print("Dropoff outside January:", dropoff_outside_january)
print("Invalid/missing pickup dates:", invalid_pickup_dates)
print("Invalid/missing dropoff dates:", invalid_dropoff_dates)

print("\nFare and distance checks")
print("Negative fares:", negative_fares)
print("Zero fares:", zero_fares)
print("Negative distances:", negative_distances)
print("Zero distances:", zero_distances)
print("Zero passenger counts:", zero_passengers)

print("\nDuplicate check")
print("Duplicate row candidates:", duplicate_rows)

print("\nMissing values by column")
print(null_counts.to_frame(name="null_count").to_string())

Total rows checked: 1369765

Date checks
Pickup outside January: 24
Dropoff outside January: 137
Invalid/missing pickup dates: 0
Invalid/missing dropoff dates: 0

Fare and distance checks
Negative fares: 6769
Zero fares: 642
Negative distances: 0
Zero distances: 19952
Zero passenger counts: 26726

Duplicate check
Duplicate row candidates: 0

Missing values by column
                       null_count
VendorID                    98352
tpep_pickup_datetime            0
tpep_dropoff_datetime           0
passenger_count             98352
trip_distance                   0
RatecodeID                  98352
store_and_fwd_flag          98352
PULocationID                    0
DOLocationID                    0
payment_type                98352
fare_amount                     0
extra                           0
mta_tax                         0
tip_amount                      0
tolls_amount                    0
improvement_surcharge           0
total_amount                    0
congestion_surcharg

The check above does not look at trips that end before they start. This cell checks dropoff before pickup, and also how many of those rows belong to the group where all five metadata columns are null. A table after it collects all the counts of Step 3.

In [4]:
metadata_columns = [
    "VendorID",
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "payment_type",
]

total_rows = 0
dropoff_before_pickup = 0
dropoff_before_pickup_in_null_group = 0
all_metadata_null = 0
fare_not_positive = 0
fare_not_positive_in_null_group = 0

for chunk in pd.read_csv(
    url,
    chunksize=chunk_size,
    dtype=dtype,
    parse_dates=parse_dates,
):
    total_rows += len(chunk)

    bad_time = chunk["tpep_dropoff_datetime"] < chunk["tpep_pickup_datetime"]
    null_group = chunk[metadata_columns].isna().all(axis=1)
    bad_fare = chunk["fare_amount"] <= 0

    dropoff_before_pickup += int(bad_time.sum())
    dropoff_before_pickup_in_null_group += int((bad_time & null_group).sum())
    all_metadata_null += int(null_group.sum())
    fare_not_positive += int(bad_fare.sum())
    fare_not_positive_in_null_group += int((bad_fare & null_group).sum())

print("Total rows checked:", f"{total_rows:,}")

print("\nDropoff before pickup")
print("  all rows                    :", f"{dropoff_before_pickup:,}")
print("  inside the null-metadata set:", f"{dropoff_before_pickup_in_null_group:,}")

print("\nRows where all 5 metadata columns are null:", f"{all_metadata_null:,}")

print("\nfare_amount <= 0")
print("  all rows                    :", f"{fare_not_positive:,}")
print("  inside the null-metadata set:", f"{fare_not_positive_in_null_group:,}")

Total rows checked: 1,369,765

Dropoff before pickup
  all rows                    : 5,642
  inside the null-metadata set: 5,640

Rows where all 5 metadata columns are null: 98,352

fare_amount <= 0
  all rows                    : 7,411
  inside the null-metadata set: 146


In [5]:
problems = pd.DataFrame({
    "problem": [
        "pickup outside January 2021",
        "dropoff outside January 2021",
        "dropoff before pickup",
        "invalid pickup/dropoff dates",
        "negative fare",
        "zero fare",
        "negative distance",
        "zero distance",
        "passenger_count = 0",
        "passenger_count is null",
        "duplicate rows",
    ],
    "rows_found": [
        pickup_outside_january,
        dropoff_outside_january,
        dropoff_before_pickup,
        invalid_pickup_dates + invalid_dropoff_dates,
        negative_fares,
        zero_fares,
        negative_distances,
        zero_distances,
        zero_passengers,
        int(null_counts["passenger_count"]),
        duplicate_rows,
    ],
})

print(problems.to_string(index=False))

                     problem  rows_found
 pickup outside January 2021          24
dropoff outside January 2021         137
       dropoff before pickup        5642
invalid pickup/dropoff dates           0
               negative fare        6769
                   zero fare         642
           negative distance           0
               zero distance       19952
         passenger_count = 0       26726
     passenger_count is null       98352
              duplicate rows           0


### Step 4: Cleaning Decisions

Each problem found in Step 3 gets one decision: drop, flag, or no action. I drop rows that are impossible or that the Lab 3 task asks to remove: null passenger_count, pickup outside January 2021, fare_amount <= 0, and dropoff before pickup. I flag rows that look unusual but could be real trips: dropoff outside January, zero distance, and passenger_count = 0.

`rows_found` comes from Step 3 and counts every row that has the problem. `effect_in_clean` is what happens in Step 5, where the rules run in order and each row is counted once, at the first rule that removes it. That is why fare_amount <= 0 shows 7,411 found but 7,264 removed: 147 of those rows were already removed by earlier rules.

In [6]:
decisions = pd.DataFrame([
    {
        "problem": "passenger_count is null",
        "rows_found": 98352,
        "decision": "Drop",
        "effect_in_clean": "removed 98,352",
        "why": "The 5 metadata columns are null together in the same 98,352 rows, so there is nothing to repair from. Lab 3 also asks for this rule.",
        "rule_owner": "Taxi Business Owner",
    },
    {
        "problem": "Pickup outside January 2021",
        "rows_found": 24,
        "decision": "Drop",
        "effect_in_clean": "removed 24",
        "why": "The file is the January 2021 data, so these trips belong to another period.",
        "rule_owner": "Data Engineering",
    },
    {
        "problem": "fare_amount <= 0",
        "rows_found": 7411,
        "decision": "Drop",
        "effect_in_clean": "removed 7,264",
        "why": "A paid trip needs a positive fare. Negative values may be refunds or adjustments that should be handled in a separate table.",
        "rule_owner": "Taxi Business Owner / Finance",
    },
    {
        "problem": "Dropoff before pickup",
        "rows_found": 5642,
        "decision": "Drop",
        "effect_in_clean": "removed 2",
        "why": "A trip cannot end before it starts. 5,640 of these rows were already removed by the null passenger_count rule.",
        "rule_owner": "Data Engineering",
    },
    {
        "problem": "Dropoff outside January 2021",
        "rows_found": 137,
        "decision": "Flag",
        "effect_in_clean": "flagged 115",
        "why": "A trip picked up on 31 January can legitimately end in February.",
        "rule_owner": "Taxi Business Owner",
    },
    {
        "problem": "Zero trip distance",
        "rows_found": 19952,
        "decision": "Flag",
        "effect_in_clean": "flagged 17,774",
        "why": "Could be a recording issue or a cancelled ride, but it does not prove the trip is invalid.",
        "rule_owner": "Taxi Business Owner",
    },
    {
        "problem": "passenger_count equals zero",
        "rows_found": 26726,
        "decision": "Flag",
        "effect_in_clean": "flagged 26,709",
        "why": "The count may be unknown or not entered. Dropping would remove about 2% of the trips.",
        "rule_owner": "Taxi Business Owner",
    },
    {
        "problem": "Negative trip distance / invalid dates / duplicate rows",
        "rows_found": 0,
        "decision": "No action",
        "effect_in_clean": "none",
        "why": "Step 3 found none of these in the file.",
        "rule_owner": "Data Engineering",
    },
])

from IPython.display import display

display(
    decisions.style
    .set_properties(**{
        "text-align": "left",
        "white-space": "normal",
        "min-width": "150px",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "left"),
                ("background-color", "#D9EAF7"),
                ("font-weight", "bold"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("padding", "8px"),
                ("vertical-align", "top"),
            ],
        },
    ])
)

,problem,rows_found,decision,effect_in_clean,why,rule_owner
0,passenger_count is null,98352,Drop,"removed 98,352","The 5 metadata columns are null together in the same 98,352 rows, so there is nothing to repair from. Lab 3 also asks for this rule.",Taxi Business Owner
1,Pickup outside January 2021,24,Drop,removed 24,"The file is the January 2021 data, so these trips belong to another period.",Data Engineering
2,fare_amount <= 0,7411,Drop,"removed 7,264",A paid trip needs a positive fare. Negative values may be refunds or adjustments that should be handled in a separate table.,Taxi Business Owner / Finance
3,Dropoff before pickup,5642,Drop,removed 2,"A trip cannot end before it starts. 5,640 of these rows were already removed by the null passenger_count rule.",Data Engineering
4,Dropoff outside January 2021,137,Flag,flagged 115,A trip picked up on 31 January can legitimately end in February.,Taxi Business Owner
5,Zero trip distance,19952,Flag,"flagged 17,774","Could be a recording issue or a cancelled ride, but it does not prove the trip is invalid.",Taxi Business Owner
6,passenger_count equals zero,26726,Flag,"flagged 26,709",The count may be unknown or not entered. Dropping would remove about 2% of the trips.,Taxi Business Owner
7,Negative trip distance / invalid dates / duplicate rows,0,No action,none,Step 3 found none of these in the file.,Data Engineering


### Step 5: Clean the Dataset

`clean(df)` works on a copy, so the original dataframe is not changed. It returns the cleaned dataframe and a dictionary with the rows removed by each rule. The rules run in this order: null passenger_count, pickup outside January 2021, fare_amount <= 0, dropoff before pickup. Three flag columns mark suspicious rows that are kept. Inside the function, assert statements check that the removed problems are really gone and that the dictionary adds up to the rows removed.

The first cell runs `clean()` on the whole file to check it and to confirm the original is unchanged. The second cell runs it chunk by chunk with the same dtype as Step 2, which is how it will be used in the pipeline. Both give the same result: 1,369,765 rows before, 1,264,123 after, 105,642 removed.

The cleaned data is not loaded into PostgreSQL.

In [7]:
import pandas as pd

url = (
    "https://github.com/DataTalksClub/nyc-tlc-data/"
    "releases/download/yellow/yellow_tripdata_2021-01.csv.gz"
)

df = pd.read_csv(
    url,
    dtype={
        "store_and_fwd_flag": "string"
    }
)

print("Full dataset shape:", df.shape)

Full dataset shape: (1369765, 18)


In [8]:
FLAG_COLUMNS = [
    "flag_dropoff_outside_january",
    "flag_zero_distance",
    "flag_zero_passengers",
]


def clean(df, verbose=True):

    # Deep copy so the original dataframe is not modified
    cleaned_df = df.copy(deep=True)

    rows_before = len(cleaned_df)
    removed_counts = {}

    # --------------------------------------------------
    # 1. Convert date columns to datetime
    # --------------------------------------------------
    cleaned_df["tpep_pickup_datetime"] = pd.to_datetime(
        cleaned_df["tpep_pickup_datetime"],
        errors="coerce"
    )
    cleaned_df["tpep_dropoff_datetime"] = pd.to_datetime(
        cleaned_df["tpep_dropoff_datetime"],
        errors="coerce"
    )

    # --------------------------------------------------
    # 2. Remove rows with null passenger_count
    # --------------------------------------------------
    null_passenger = cleaned_df["passenger_count"].isna()

    removed_counts["null_passenger_count"] = int(
        null_passenger.sum()
    )

    cleaned_df = cleaned_df.loc[
        ~null_passenger
    ].copy()

    # --------------------------------------------------
    # 3. Remove pickup dates outside January 2021
    # --------------------------------------------------
    january_start = pd.Timestamp("2021-01-01")
    february_start = pd.Timestamp("2021-02-01")

    pickup_outside_january = (
        (cleaned_df["tpep_pickup_datetime"] < january_start)
        |
        (cleaned_df["tpep_pickup_datetime"] >= february_start)
    )

    removed_counts["pickup_outside_january"] = int(
        pickup_outside_january.sum()
    )

    cleaned_df = cleaned_df.loc[
        ~pickup_outside_january
    ].copy()

    # --------------------------------------------------
    # 4. Remove rows with fare_amount <= 0
    # --------------------------------------------------
    bad_fare = cleaned_df["fare_amount"] <= 0

    removed_counts["fare_amount_not_positive"] = int(
        bad_fare.sum()
    )

    cleaned_df = cleaned_df.loc[
        ~bad_fare
    ].copy()

    # --------------------------------------------------
    # 5. Remove trips that end before they start
    # --------------------------------------------------
    dropoff_before_pickup = (
        cleaned_df["tpep_dropoff_datetime"]
        < cleaned_df["tpep_pickup_datetime"]
    )

    removed_counts["dropoff_before_pickup"] = int(
        dropoff_before_pickup.sum()
    )

    cleaned_df = cleaned_df.loc[
        ~dropoff_before_pickup
    ].copy()

    # --------------------------------------------------
    # 6. Flags: suspicious rows that we keep
    # --------------------------------------------------
    cleaned_df["flag_dropoff_outside_january"] = (
        (cleaned_df["tpep_dropoff_datetime"] < january_start)
        |
        (cleaned_df["tpep_dropoff_datetime"] >= february_start)
    )
    cleaned_df["flag_zero_distance"] = cleaned_df["trip_distance"] == 0
    cleaned_df["flag_zero_passengers"] = cleaned_df["passenger_count"] == 0

    # --------------------------------------------------
    # 7. Summary and verification
    # --------------------------------------------------
    rows_after = len(cleaned_df)
    total_removed = rows_before - rows_after

    assert total_removed == sum(removed_counts.values())

    # Verify that the problems we drop are really gone
    assert cleaned_df["passenger_count"].notna().all()

    assert (
        (cleaned_df["tpep_pickup_datetime"] >= january_start)
        &
        (cleaned_df["tpep_pickup_datetime"] < february_start)
    ).all()

    assert (cleaned_df["fare_amount"] > 0).all()

    assert (
        cleaned_df["tpep_dropoff_datetime"]
        >= cleaned_df["tpep_pickup_datetime"]
    ).all()

    if verbose:
        print("Rows before cleaning:", f"{rows_before:,}")
        print("Rows after cleaning :", f"{rows_after:,}")
        print("Total rows removed  :", f"{total_removed:,}")

        print("\nRows removed per rule:")

        for rule, count in removed_counts.items():
            print(f"  {rule:<30}{count:>10,}")

        print("\nRows flagged but kept:")

        for col in FLAG_COLUMNS:
            print(f"  {col:<30}{int(cleaned_df[col].sum()):>10,}")

    return cleaned_df, removed_counts


before = df.copy(deep=True)

cleaned_df, removed_counts = clean(df)

print("\nOriginal unchanged:", df.equals(before))
print("Original shape:", df.shape)
print("Cleaned shape:", cleaned_df.shape)

Rows before cleaning: 1,369,765
Rows after cleaning : 1,264,123
Total rows removed  : 105,642

Rows removed per rule:
  null_passenger_count              98,352
  pickup_outside_january                24
  fare_amount_not_positive           7,264
  dropoff_before_pickup                  2

Rows flagged but kept:
  flag_dropoff_outside_january         115
  flag_zero_distance                17,774
  flag_zero_passengers              26,709

Original unchanged: True
Original shape: (1369765, 18)
Cleaned shape: (1264123, 21)


In [9]:
from collections import Counter

total_before = 0
total_after = 0
removed_total = Counter()
flagged_total = Counter()

for chunk in pd.read_csv(
    url,
    chunksize= chunk_size,
    dtype=dtype,
    parse_dates=parse_dates,
):
    chunk_clean, removed = clean(chunk, verbose=False)

    total_before += len(chunk)
    total_after += len(chunk_clean)
    removed_total.update(removed)

    for col in FLAG_COLUMNS:
        flagged_total[col] += int(chunk_clean[col].sum())

print("Rows before cleaning:", f"{total_before:,}")
print("Rows after cleaning :", f"{total_after:,}")
print("Total rows removed  :", f"{total_before - total_after:,}")

print("\nRows removed per rule:")
for rule, count in removed_total.items():
    print(f"  {rule:<30}{count:>10,}")

print("\nRows flagged but kept:")
for col, count in flagged_total.items():
    print(f"  {col:<30}{count:>10,}")

assert total_before == 1_369_765
assert total_before - total_after == sum(removed_total.values())
print("\nChecks passed: row count matches class number and the dict adds up")

Rows before cleaning: 1,369,765
Rows after cleaning : 1,264,123
Total rows removed  : 105,642

Rows removed per rule:
  null_passenger_count              98,352
  pickup_outside_january                24
  fare_amount_not_positive           7,264
  dropoff_before_pickup                  2

Rows flagged but kept:
  flag_dropoff_outside_january         115
  flag_zero_distance                17,774
  flag_zero_passengers              26,709

Checks passed: row count matches class number and the dict adds up
